# 20 — End-to-end Göttingen demo
Connect demo sales → contextual features → forecast → inventory plan. This notebook is the compact reference workflow for the repository.

In [ ]:
import pandas as pd
from demand_supply.features import enrich_daily_sales
from demand_supply.forecasting.prophet_model import ProphetDemandModel
from demand_supply.forecasting.inventory import inventory_plan
df=pd.read_parquet('../data/processed/demo_daily_sales.parquet')
sku=df.query("product_id == 'MILK-1L'").copy().sort_values('ds')
data=enrich_daily_sales(sku)


In [ ]:
reg=['event_lift','is_public_holiday','promotion','population_share']
for c in reg:
    if c in data: data[c]=data[c].fillna(0).astype(float)
train=data.iloc[:-28]; future=data.iloc[-28:]
model=ProphetDemandModel(regressors=reg).fit(train)
forecast=model.predict(future)
forecast.head()


In [ ]:
plan=inventory_plan(forecast,on_hand=120,on_order=40,lead_time_days=3,daily_std=float(train.y.std()),service_level=.95)
plan


## Production checklist
1. Replace demo sales with retailer POS/ERP feeds.
2. Archive historical weather and calibrate event effects.
3. Backtest each SKU/category and keep a seasonal-naive champion/challenger.
4. Add supplier constraints and perishable-life rules.
5. Deploy API/dashboard behind TLS and OIDC.
6. Monitor forecast + business KPIs continuously.
